In [32]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [33]:
model = SentenceTransformer('all-MiniLM-L6-v2')

## Balanced Dataset

In [34]:
df_bal = pd.read_csv(r"dataset\preprocessing\prePro-news_balanced.csv")
print(f'''
Shape: {df_bal.shape}
Columns: {df_bal.columns.tolist()}

{df_bal.category.value_counts()}
''')
df_bal.head()


Shape: (1291, 4)
Columns: ['content_id', 'sentence', 'published_date', 'category']

Gadgets    883
5G         408
Name: category, dtype: int64



,content_id,sentence,published_date,category
0,1442420,"Until now, the AirPods Pro were all about keep...",2024-09-17,Gadgets
1,1442420,Apple says the latest AirPods Pro 2 can be use...,2024-09-17,Gadgets
2,1442420,"Pending approval by regulatory authorities, th...",2024-09-17,Gadgets
3,1442420,The company is also planning to integrate a he...,2024-09-17,Gadgets
4,1442420,"If the feature takes hold, Apple could use it ...",2024-09-17,Gadgets


## Sentence Embeddings (Balanced)

In [35]:
sentences = df_bal['sentence'].tolist()

sentence_embeddings = model.encode(sentences, show_progress_bar=True)

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

In [37]:
print("Sentence Embedding Shape:", sentence_embeddings.shape)

Sentence Embedding Shape: (1291, 384)


## Timestamp Embeddings (Balanced)

In [38]:
df_bal['timestamp'] = pd.to_datetime(df_bal['published_date'])

In [48]:
t0 = df_bal['timestamp'].min()

df_bal['t'] = (df_bal['timestamp'] - t0).dt.days
df_bal['t_norm'] = df_bal['t'] / df_bal['t'].max()

In [52]:
time_embeddings = df_bal['t_norm'].values.reshape(-1,1)

print(time_embeddings.shape)

(1291, 1)


## Combining both the Embeddings (Balanced)

In [57]:
final_embeddings_bal = np.hstack([
    sentence_embeddings,
    time_embeddings
])

print(final_embeddings_bal.shape)

(1291, 385)


## Un-balanced Dataset

In [58]:
df_unbal = pd.read_csv(r"dataset\preprocessing\prePro-news_filtered.csv")
print(f'''
Shape: {df_unbal.shape}
Columns: {df_unbal.columns.tolist()}

{df_unbal.category.value_counts()}
''')
df_unbal.head()


Shape: (5120, 4)
Columns: ['content_id', 'sentence', 'published_date', 'category']

Gadgets    4712
5G          408
Name: category, dtype: int64



,content_id,sentence,published_date,category
0,1808882,CelcomDigi has refreshed its Postpaid 5G plans...,2026-02-05,5G
1,1808882,The company said the newest addition is the Po...,2026-02-05,5G
2,1808882,The plan also includes access to streaming ser...,2026-02-05,5G
3,1808882,Other unlimited data with uncapped speed plans...,2026-02-05,5G
4,1808882,The unlimited quota is subject to the company'...,2026-02-05,5G


## Sentence Embeddings (Un-Balanced)

In [60]:
sentences = df_unbal['sentence'].tolist()

sentence_embeddings = model.encode(sentences, show_progress_bar=True)
print("Sentence Embedding Shape:", sentence_embeddings.shape)

Batches:   0%|          | 0/160 [00:00<?, ?it/s]

Sentence Embedding Shape: (5120, 384)


## Timestamp Embeddings (Balanced)

In [61]:
df_unbal['timestamp'] = pd.to_datetime(df_unbal['published_date'])
t0 = df_unbal['timestamp'].min()

df_unbal['t'] = (df_unbal['timestamp'] - t0).dt.days
df_unbal['t_norm'] = df_unbal['t'] / df_unbal['t'].max()
time_embeddings = df_unbal['t_norm'].values.reshape(-1,1)

print(time_embeddings.shape)

(5120, 1)


## Combining both the Embeddings (Balanced)

In [62]:
final_embeddings_unbal = np.hstack([
    sentence_embeddings,
    time_embeddings
])

print(final_embeddings_unbal.shape)

(5120, 385)
